# Hybrid Feature Fusion OCR Framework
This notebook implements a simplified hybrid OCR pipeline


In [13]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import pytesseract

## Configuration

In [14]:
DATASET_PATH = 'D:/ABC_Project/BMVC_image_data/blur'
IMAGE_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 5

# Test dataset path
TEST_DATASET_PATH = 'D:/ABC_Project/BMVC_OCR_test_data'


## Image Preprocessing

In [15]:
def remove_noise(img):
    return cv2.medianBlur(img, 3)

def enhance_contrast(img):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    return clahe.apply(img)

def preprocess(img):
    img = remove_noise(img)
    img = enhance_contrast(img)
    img = cv2.resize(img,(IMAGE_SIZE,IMAGE_SIZE))
    return img


## Frequency Domain Features (FFT)

In [16]:
def extract_frequency_features(image):
    f = np.fft.fft2(image)
    fshift = np.fft.fftshift(f)
    magnitude = np.log(np.abs(fshift) + 1)
    features = cv2.resize(magnitude,(32,32)).flatten()
    return features


## Dataset Loader

In [17]:
import pickle
from pathlib import Path

cache_dir = Path('D:/ABC_Project/cache')
cache_dir.mkdir(exist_ok=True)

def precache_dataset(image_dir, cache_dir):
    """Pre-compute and cache all images once"""
    cache_file = cache_dir / 'dataset_cache.pkl'
    
    if cache_file.exists():
        print(f"Loading cached dataset...")
        with open(cache_file, 'rb') as f:
            return pickle.load(f)
    
    print(f"Pre-computing all images (first time only)...")
    images = os.listdir(image_dir)
    print(f"Found {len(images)} images")
    
    cache_data = {}
    
    for i, img_name in enumerate(tqdm(images)):
        if i % 100 == 0:
            print(f"  Processing {i}/{len(images)}...")
        
        try:
            img_path = os.path.join(image_dir, img_name)
            img = cv2.imread(img_path)
            
            if img is None:
                print(f"  Warning: Could not read {img_name}")
                continue
                
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            img = preprocess(img)
            freq_features = extract_frequency_features(img)
            img = img / 255.0
            
            cache_data[img_name] = {
                'img': torch.tensor(img).float().unsqueeze(0),
                'freq_features': torch.tensor(freq_features).float()
            }
        except Exception as e:
            print(f"  Error processing {img_name}: {e}")
            continue
    
    print(f"Saving cache to {cache_file}...")
    with open(cache_file, 'wb') as f:
        pickle.dump(cache_data, f)
    
    print(f"Cached {len(cache_data)} images")
    return cache_data

class OCRDataset(Dataset):
    def __init__(self, image_dir, use_cache=True):
        self.image_dir = image_dir
        self.images = os.listdir(image_dir)
        self.use_cache = use_cache
        self.cache = precache_dataset(image_dir, cache_dir) if use_cache else None

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        
        if self.use_cache:
            # Load from cache (instant, no preprocessing!)
            data = self.cache[img_name]
            img = data['img']
            freq_features = data['freq_features']
        else:
            # Original on-the-fly processing
            img_path = os.path.join(self.image_dir, img_name)
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            img = preprocess(img)
            freq_features = extract_frequency_features(img)
            img = img / 255.0
            img = torch.tensor(img).float().unsqueeze(0)
            freq_features = torch.tensor(freq_features).float()

        label = torch.randint(0,36,(1,)).item()
        return img, freq_features, label


## CNN Spatial Feature Extractor

In [18]:
class CNNFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.flatten = nn.Flatten()

    def forward(self,x):
        x = self.conv(x)
        x = self.flatten(x)
        return x


## Hybrid Feature Fusion Model

In [19]:
class HybridOCR(nn.Module):
    def __init__(self, cnn_feature_size, freq_feature_size):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(cnn_feature_size + freq_feature_size,512),
            nn.ReLU(),
            nn.Linear(512,128),
            nn.ReLU(),
            nn.Linear(128,36)
        )

    def forward(self,cnn_feat,freq_feat):
        x = torch.cat((cnn_feat,freq_feat),dim=1)
        return self.fc(x)


## Training Loop


In [20]:
def train_model(cnn_model, hybrid_model, dataloader, device):
    criterion = nn.CrossEntropyLoss()
    # Optimize both CNN and hybrid model parameters
    optimizer = optim.Adam(list(cnn_model.parameters()) + list(hybrid_model.parameters()), lr=0.001)

    for epoch in range(EPOCHS):
        total_loss = 0

        for images, freq_features, labels in tqdm(dataloader):
            images = images.to(device)
            freq_features = freq_features.to(device)
            labels = labels.to(device)
            
            cnn_features = cnn_model(images)
            outputs = hybrid_model(cnn_features, freq_features)

            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print('Epoch:', epoch+1, 'Loss:', total_loss)

## Run Pipeline

In [21]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


Using device: cuda
NVIDIA GeForce RTX 5060 Laptop GPU


In [22]:
dataset = OCRDataset(DATASET_PATH, use_cache=False)  # Enables dataset caching
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)

cnn_model = CNNFeatureExtractor().to(device)

dummy = torch.randn(1,1,IMAGE_SIZE,IMAGE_SIZE).to(device)
cnn_feature_size = cnn_model(dummy).shape[1]

freq_feature_size = 1024

hybrid_model = HybridOCR(cnn_feature_size,freq_feature_size).to(device)

# Print device information
import platform
device_type = 'GPU' if device.type == 'cuda' else 'CPU'
if device.type == 'cuda':
    device_name = torch.cuda.get_device_name(0)
else:
    device_name = platform.processor()

print(f'\n{"="*60}')
print(f'Training Configuration:')
print(f'Device: {device_type}')
print(f'Device Name: {device_name}')
print(f'Dataset: {DATASET_PATH}')
print(f'Batch Size: {BATCH_SIZE}')
print(f'Epochs: {EPOCHS}')
print(f'Image Size: {IMAGE_SIZE}')
print(f'Workers: 2')
print(f'Caching: Enabled')
print(f'{"="*60}\n')

# Verify models are on correct device before training
print("Device Assignment Check:")
print(f"Device: {device}")
print(f"CNN Model device: {next(cnn_model.parameters()).device}")
print(f"Hybrid Model device: {next(hybrid_model.parameters()).device}")

# For PyTorch 2.0+ with CUDA, optionally enable TF32 for faster computation
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    print("CUDA optimizations enabled (TF32, cuDNN benchmark)\n")





Training Configuration:
Device: GPU
Device Name: NVIDIA GeForce RTX 5060 Laptop GPU
Dataset: D:/ABC_Project/BMVC_image_data/blur
Batch Size: 32
Epochs: 5
Image Size: 256
Workers: 2
Caching: Enabled

Device Assignment Check:
Device: cuda
CNN Model device: cuda:0
Hybrid Model device: cuda:0
CUDA optimizations enabled (TF32, cuDNN benchmark)



In [23]:
train_model(cnn_model,hybrid_model,loader,device)

100%|██████████| 2086/2086 [14:45<00:00,  2.36it/s]


Epoch: 1 Loss: 7488.337531328201


100%|██████████| 2086/2086 [15:53<00:00,  2.19it/s]


Epoch: 2 Loss: 7475.698323011398


100%|██████████| 2086/2086 [16:29<00:00,  2.11it/s]


Epoch: 3 Loss: 7475.857954978943


100%|██████████| 2086/2086 [17:21<00:00,  2.00it/s]


Epoch: 4 Loss: 7475.733102083206


100%|██████████| 2086/2086 [17:47<00:00,  1.95it/s]

Epoch: 5 Loss: 7475.89066362381


## OCR Test using Tesseract

In [24]:
sample = os.listdir(DATASET_PATH)[0]
img_path = os.path.join(DATASET_PATH,sample)
img = cv2.imread(img_path)
text = pytesseract.image_to_string(img)
print('Recognized text:')
print(text)

Recognized text:
‘Customary +
pects of the vMF lobe and therefore cann
ted. To solve this problem, we recall fron
can be inferred from the scaled Euclidean
ven VMF distribution. By lincarity of expe
late Gar = (x) linearly, as well as the amplit

& = T(a;) t=Tlas,
vere T(.) denotes trilincar hardware
Can canily be found in-shader (lines 9 and
Algonthm 2 shows pacudocode for our Gl
pcs 5-10 look up @ and ar, and then comp:
mentation. we store the jth lobe of cach n
IBA MIP-map (wMF Texture in Algorithen |

and enc channd! cock for Ger Gane one,

